**Evaluating a Boolean circuit with matrix algebra.**
This notebook applies the gate-matrix formalism from `gate_matrices.ipynb`
to a concrete four-input circuit.
The circuit implements:

$$\text{out} = \lnot\,\bigl[(\,a \land b\,) \lor (\lnot c)\bigr] \land d$$

Each gate is a matrix; each Boolean value is a one-hot column vector.
The Kronecker product $\mathbf{x} \otimes \mathbf{y}$ packs two one-hot
vectors into the four-element input that AND and OR expect,
and **composing gates is matrix multiplication**.
After tracing one input combination through every stage,
the full 16-row truth table is generated and verified against the
equivalent Python boolean expression.

In [ ]:
"""simple_circuit.ipynb"""

# Cell 01 - Imports and common Boolean states and gates

import numpy as np
from IPython.display import Math, display

from qis101_utils import as_latex

# One-hot (indicator) vectors: the position of the 1 encodes the state.
# F = |0⟩ = [1, 0]ᵀ  (index 0 = False)
# T = |1⟩ = [0, 1]ᵀ  (index 1 = True)
f = np.array([[1], [0]])
t = np.array([[0], [1]])

g_not = np.array([[0, 1], [1, 0]])
g_and = np.array([[1, 1, 1, 0], [0, 0, 0, 1]])
g_or = np.array([[1, 0, 0, 0], [0, 1, 1, 1]])

display(as_latex(f, prefix=r"\mathbf{F}=0="))
display(as_latex(t, prefix=r"\mathbf{T}=1="))
display(as_latex(g_not, prefix=r"\mathbf{NOT}="))
display(as_latex(g_and, prefix=r"\mathbf{AND}="))
display(as_latex(g_or, prefix=r"\mathbf{OR}="))

**Circuit topology.**
Each internal signal is labelled $g_1 \ldots g_5$:

```
 a ──┐
     AND ── g1 ──┐
 b ──┘            OR ── g3 ──┐
                 /            AND ── g4 ── NOT ── out
 c ── NOT ── g2 ┘            /
                        d ──┘
```

Signal definitions:

| Signal | Expression |
|---|---|
| $g_1$ | $a \land b$ |
| $g_2$ | $\lnot c$ |
| $g_3$ | $g_1 \lor g_2 = (a \land b) \lor (\lnot c)$ |
| $g_4$ | $g_3 \land d$ |
| out | $\lnot\, g_4$ |

In [ ]:
# Cell 02 - Helper: convert a one-hot column vector to a readable symbol


def vec2sym(v: np.ndarray) -> str:
    """Return 'T' or 'F' for a one-hot column vector."""
    return "T" if v[1, 0] == 1 else "F"


def vec2bool(v: np.ndarray) -> bool:
    """Convert a one-hot column vector to a Python bool."""
    return bool(v[1, 0])

In [ ]:
# Cell 03 - Trace one representative input through every gate stage
#           Input: a=T, b=T, c=T, d=T  ->  expected output: F

a, b, c, d = t, t, t, t

g1 = g_and @ np.kron(a, b)  # g1 = a AND b
g2 = g_not @ c  # g2 = NOT c
g3 = g_or @ np.kron(g1, g2)  # g3 = g1 OR g2  =  (a AND b) OR (NOT c)
g4 = g_and @ np.kron(g3, d)  # g4 = g3 AND d
g5 = g_not @ g4  # g5 = NOT g4  =  output

display(Math(rf"a={vec2sym(a)},\; b={vec2sym(b)},\; c={vec2sym(c)},\; d={vec2sym(d)}"))
display(as_latex(g1, prefix=r"g_1 = a \land b = "))
display(as_latex(g2, prefix=r"g_2 = \lnot\, c = "))
display(as_latex(g3, prefix=r"g_3 = g_1 \lor g_2 = "))
display(as_latex(g4, prefix=r"g_4 = g_3 \land d = "))
display(as_latex(g5, prefix=r"\mathrm{out} = \lnot\, g_4 = "))

In [ ]:
# Cell 04 - Full truth table across all 16 input combinations


def circuit(a: np.ndarray, b: np.ndarray, c: np.ndarray, d: np.ndarray) -> np.ndarray:
    """Evaluate the Boolean circuit for inputs a, b, c, d.

    Circuit logic (using matrix algebra):
        g1 = a AND b
        g2 = NOT c
        g3 = g1 OR g2      ->  (a AND b) OR (NOT c)
        g4 = g3 AND d
        out = NOT g4       ->  NOT( ((a AND b) OR (NOT c)) AND d )

    Parameters
    ----------
    a, b, c, d : ndarray, shape (2, 1)
        One-hot column vectors representing Boolean inputs.

    Returns
    -------
    ndarray, shape (2, 1)
        One-hot column vector representing the circuit output.
    """
    g1 = g_and @ np.kron(a, b)
    g2 = g_not @ c
    g3 = g_or @ np.kron(g1, g2)
    g4 = g_and @ np.kron(g3, d)
    return g_not @ g4


print(f"{'a':>3} {'b':>3} {'c':>3} {'d':>3}  ->  out")
print("-" * 25)
for a in [f, t]:
    for b in [f, t]:
        for c in [f, t]:
            for d in [f, t]:
                out = circuit(a, b, c, d)
                print(
                    f"  {vec2sym(a)}   {vec2sym(b)}   {vec2sym(c)}   {vec2sym(d)}"
                    f"  ->   {vec2sym(out)}"
                )

**Verification against pure Python.**
Each row of the truth table can be independently checked against the
equivalent Python boolean expression:

```python
not (((a and b) or (not c)) and d)
```

The cell below confirms that the matrix circuit agrees with this
expression for all 16 input combinations.

In [ ]:
# Cell 05 - Verify matrix circuit against Python boolean expression for all 16 inputs

all_pass = True
for a in [f, t]:
    for b in [f, t]:
        for c in [f, t]:
            for d in [f, t]:
                ab, bb, cb, db = vec2bool(a), vec2bool(b), vec2bool(c), vec2bool(d)
                expected = not (((ab and bb) or (not cb)) and db)
                got = vec2bool(circuit(a, b, c, d))
                if expected != got:
                    print(
                        f"MISMATCH  a={ab} b={bb} c={cb} d={db}: "
                        f"expected={expected}, got={got}"
                    )
                    all_pass = False

symbol = r"\checkmark" if all_pass else r"\times"
display(
    Math(
        rf"{symbol}\; \text{{All 16 rows match }}"
        rf"\lnot\,[\,(a \land b) \lor (\lnot c)\,] \land d"
    )
)